In [3]:
%load_ext sql

In [5]:
%sql sqlite:///sales.db

In [8]:
%%sql
SELECT * FROM Orders;

 * sqlite:///sales.db
Done.


order_id,product_id,quantity,region
101,1,2,North
102,2,5,South
103,3,3,North
104,1,1,East
105,2,4,South
106,3,6,West


In [9]:
%%sql
select * from Products

 * sqlite:///sales.db
Done.


product_id,product_name,unit_price
1,Laptop,800.0
2,Mouse,20.0
3,Keyboard,50.0


In [15]:
%%sql
select Orders.region, 
sum(Orders.quantity*Products.unit_price) as total_revenue from Orders, Products
where Orders.product_id=Products.product_id
group by region;

 * sqlite:///sales.db
Done.


region,total_revenue
East,800.0
North,1750.0
South,180.0
West,300.0


In [23]:
%%sql
select order_id, Orders.region, product_name, 
    Orders.quantity*Products.unit_price as order_revenue,
    sum(Orders.quantity*Products.unit_price) over (partition by region) 
    as regional_revenue from Orders, Products
where Orders.product_id=Products.product_id;

 * sqlite:///sales.db
Done.


order_id,region,product_name,order_revenue,regional_revenue
104,East,Laptop,800.0,800.0
101,North,Laptop,1600.0,1750.0
103,North,Keyboard,150.0,1750.0
102,South,Mouse,100.0,180.0
105,South,Mouse,80.0,180.0
106,West,Keyboard,300.0,300.0


In [24]:
%%sql
select order_id, Orders.region, product_name, 
    Orders.quantity*Products.unit_price as order_revenue,
    sum(Orders.quantity*Products.unit_price) over (partition by region) 
    as regional_revenue from Orders
    join Products
    on Orders.product_id=Products.product_id;

 * sqlite:///sales.db
Done.


order_id,region,product_name,order_revenue,regional_revenue
104,East,Laptop,800.0,800.0
101,North,Laptop,1600.0,1750.0
103,North,Keyboard,150.0,1750.0
102,South,Mouse,100.0,180.0
105,South,Mouse,80.0,180.0
106,West,Keyboard,300.0,300.0


In [33]:
%%sql
select order_id, region,product_name,
    Orders.quantity*Products.unit_price as order_revenue,
    rank() over (Partition by region order by Orders.quantity*Products.unit_price DESC)
    as revenue_rank
    from Orders
    join Products
    on Orders.product_id=Products.product_id;

 * sqlite:///sales.db
Done.


order_id,region,product_name,order_revenue,revenue_rank
104,East,Laptop,800.0,1
101,North,Laptop,1600.0,1
103,North,Keyboard,150.0,2
102,South,Mouse,100.0,1
105,South,Mouse,80.0,2
106,West,Keyboard,300.0,1


In [37]:
%%sql

CREATE TABLE EmployeeSales (
    employee_id INTEGER,
    employee_name TEXT,
    department TEXT,
    sales REAL
);

INSERT INTO EmployeeSales VALUES
(1, 'Alice', 'North', 1200),
(2, 'Bob', 'North', 1000),
(3, 'Carol', 'North', 1000),
(4, 'David', 'North', 800),
(5, 'Eva', 'South', 1500),
(6, 'Frank', 'South', 1500),
(7, 'Grace', 'South', 1100),
(8, 'Henry', 'South', 900);

 * sqlite:///sales.db
Done.
8 rows affected.


[]

In [38]:
%%sql
select * from EmployeeSales

 * sqlite:///sales.db
Done.


employee_id,employee_name,department,sales
1,Alice,North,1200.0
2,Bob,North,1000.0
3,Carol,North,1000.0
4,David,North,800.0
5,Eva,South,1500.0
6,Frank,South,1500.0
7,Grace,South,1100.0
8,Henry,South,900.0


In [43]:
%%sql
select employee_name, department,sales,
    rank() over (partition by department order by sales desc) as rank,
    row_number() over (partition by department order by sales desc) as rank_num,
    dense_rank() over (partition by department order by sales desc) as rank_dense
    from EmployeeSales;


 * sqlite:///sales.db
Done.


employee_name,department,sales,rank,rank_num,rank_dense
Alice,North,1200.0,1,1,1
Bob,North,1000.0,2,2,2
Carol,North,1000.0,2,3,2
David,North,800.0,4,4,3
Eva,South,1500.0,1,1,1
Frank,South,1500.0,1,2,1
Grace,South,1100.0,3,3,2
Henry,South,900.0,4,4,3


In [44]:
%%sql
select employee_name, department,sales,
    dense_rank() over (partition by department order by sales asc) as rank_dense
    from EmployeeSales;

 * sqlite:///sales.db
Done.


employee_name,department,sales,rank_dense
David,North,800.0,1
Bob,North,1000.0,2
Carol,North,1000.0,2
Alice,North,1200.0,3
Henry,South,900.0,1
Grace,South,1100.0,2
Eva,South,1500.0,3
Frank,South,1500.0,3


In [50]:
%%sql
select employee_name, department,sales from EmployeeSales
    where sales>(select avg(sales) from EmployeeSales);


 * sqlite:///sales.db
Done.


employee_name,department,sales
Alice,North,1200.0
Eva,South,1500.0
Frank,South,1500.0


In [53]:
%%sql
with average_sales as (
    select avg(sales) as avg_sales from EmployeeSales)
select employee_name, department, sales
FROM EmployeeSales, average_sales
 WHERE sales > avg_sales;

 * sqlite:///sales.db
Done.


employee_name,department,sales
Alice,North,1200.0
Eva,South,1500.0
Frank,South,1500.0


In [67]:
%%sql
select department, sum(sales) as total_sales,
    rank() over (order by sum(sales) desc) as sales_rank
    from EmployeeSales
    group by department;

 * sqlite:///sales.db
Done.


department,total_sales,sales_rank
South,5000.0,1
North,4000.0,2


In [74]:
%%sql
with department_summary as 
    (select department, sum(sales) as total_sales
    from EmployeeSales group by department)
    select department, total_sales,
    rank() over (order by total_sales desc) as sales_rank
    from  department_summary;


 * sqlite:///sales.db
Done.


department,total_sales,sales_rank
South,5000.0,1
North,4000.0,2
